In [47]:
# Core Imports
import os
from dotenv import load_dotenv
load_dotenv()

# LangChain & Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_unstructured import UnstructuredLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Optional, Literal

# Memory imports
from langchain.memory import ConversationBufferWindowMemory
from langchain.schema import BaseMessage, HumanMessage, AIMessage


## Document Loading and Processing


In [48]:
# Load document
file_path = "test.pdf"

if file_path.endswith(".pdf"):
    loader = PyPDFLoader(file_path)
elif file_path.endswith(".txt"):
    loader = TextLoader(file_path)
else:
    loader = UnstructuredLoader(file_path)

documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(f"First document preview:\n{documents[0].page_content[:500]}")


Loaded 14 documents
First document preview:
1: TPS Notes|| notes.tulsiprasad.com.np  
 
Unit-3: Introduction to Management Information System 
Data, information, computer based information system (CBIS), Information System Resources, Management 
Information System, Transaction Processing System(TPS), Decision Support System (DSS), Executive 
Information System (EIS), SCM, CRM and International Syst em: Introduction, Supply Chain Management 
Systems, Customer Relationships Management System, Enterprise System and Challenges of Enterprise 



In [49]:
# Text splitting
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

print(f"Split into {len(docs)} chunks")
print(f"First chunk preview:\n{docs[0].page_content[:500]}")


Split into 54 chunks
First chunk preview:
1: TPS Notes|| notes.tulsiprasad.com.np  
 
Unit-3: Introduction to Management Information System 
Data, information, computer based information system (CBIS), Information System Resources, Management 
Information System, Transaction Processing System(TPS), Decision Support System (DSS), Executive 
Information System (EIS), SCM, CRM and International Syst em: Introduction, Supply Chain Management 
Systems, Customer Relationships Management System, Enterprise System and Challenges of Enterprise 



In [50]:
# Embeddings
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)


In [51]:
# Vector store
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory="chroma_db"
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Chroma vector store created.")


Chroma vector store created.


## Model and Tools Setup


In [52]:
# LLM setup
load_dotenv()
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

# Output parser
parser = StrOutputParser()


In [53]:
# Web search tool
serp_api_key = os.getenv("SERPAPI_API_KEY")
serp = SerpAPIWrapper(serpapi_api_key=serp_api_key)

search_tool = Tool(
    name="Search",
    func=serp.run,
    description="Search web queries when answer not found in documents"
)

print("Web search tool configured.")


Web search tool configured.


## Memory Setup


In [54]:
# Initialize ConversationBufferWindowMemory
memory = ConversationBufferWindowMemory(
    k=6,  # Keep last 6 messages (3 exchanges)
    memory_key="chat_history",
    return_messages=True,
    input_key="query",
    output_key="answer"
)

print("ConversationBufferWindowMemory initialized with window size of 6 messages")


ConversationBufferWindowMemory initialized with window size of 6 messages


C:\Users\Acer\AppData\Local\Temp\ipykernel_12448\3896926736.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


## Graph State Definition


In [55]:
# Graph state with memory support
class GraphState(TypedDict):
    query: str
    docs: Optional[List[Document]]
    answer: Optional[str]
    source: Optional[str]
    use_rag: Optional[bool]
    chat_history: Optional[List[BaseMessage]]


## Node Functions


In [56]:
def retriever_node(state: GraphState):
    """Retrieve relevant documents from vector store with memory context"""
    query = state["query"]
    chat_history = state.get("chat_history", [])
    
    # Enhance query with recent context if available
    enhanced_query = query
    if chat_history:
        recent_context = []
        for msg in chat_history[-4:]:
            if isinstance(msg, HumanMessage):
                recent_context.append(f"Previous question: {msg.content}")
            elif isinstance(msg, AIMessage):
                recent_context.append(f"Previous answer: {msg.content[:100]}...")
        
        if recent_context:
            enhanced_query = f"{query} (Context: {' '.join(recent_context)})"
    
    results = vector_store.similarity_search_with_score(enhanced_query, k=5)
    
    # Filter by distance threshold
    max_distance = 1.0
    filtered_docs = [doc for doc, score in results if score < max_distance]
    
    print(f"Retriever found {len(filtered_docs)} relevant docs from {len(results)} total")
    
    use_rag = len(filtered_docs) > 0
    
    return {
        "docs": filtered_docs,
        "use_rag": use_rag
    }


In [57]:
def rag_node(state: GraphState):
    """Answer question using retrieved documents and conversation history"""
    docs = state.get("docs", [])
    query = state["query"]
    chat_history = state.get("chat_history", [])
    
    if not docs:
        return {"answer": "No relevant documents found.", "source": "None"}
    
    context = "\n\n".join([d.page_content for d in docs])
    
    # Format chat history for context
    history_context = ""
    if chat_history:
        history_context = "\n\nPrevious conversation:\n"
        for msg in chat_history[-4:]:
            if isinstance(msg, HumanMessage):
                history_context += f"Human: {msg.content}\n"
            elif isinstance(msg, AIMessage):
                history_context += f"Assistant: {msg.content}\n"
    
    prompt = ChatPromptTemplate.from_template(
        """You are an intelligent assistant with access to conversation history. 
        Use the provided context and conversation history to answer the question.
        If the answer can be found in the context, provide a clear and accurate response.
        If the answer is not present in the context, respond: "Not found in document".
        
        {history_context}
        
        Context from documents:
        {context}

        Current Question: {question}

        Answer:"""
    )
    
    chain = prompt | llm | parser
    answer = chain.invoke({
        "context": context, 
        "question": query,
        "history_context": history_context
    })
    
    sources = list(set([d.metadata.get("source", "Unknown") for d in docs]))
    sources_str = ", ".join(sources)
    
    print(f"RAG answered using {len(docs)} documents with memory context")
    
    return {
        "answer": answer,
        "source": f"RAG (Documents: {sources_str})"
    }


In [58]:
def web_search_node(state: GraphState):
    """Search the web when documents don't have the answer, with memory context"""
    query = state["query"]
    chat_history = state.get("chat_history", [])
    
    # Format chat history for context
    history_context = ""
    if chat_history:
        history_context = "\n\nPrevious conversation:\n"
        for msg in chat_history[-4:]:
            if isinstance(msg, HumanMessage):
                history_context += f"Human: {msg.content}\n"
            elif isinstance(msg, AIMessage):
                history_context += f"Assistant: {msg.content}\n"
    
    # Enhance query with context if available
    enhanced_query = query
    if history_context:
        enhanced_query = f"{query} (Context: {history_context})"
    
    print("Executing web search with memory context...")
    result = search_tool.func(enhanced_query)
    
    return {
        "answer": result,
        "source": "Web Search (SerpAPI)"
    }


In [59]:
def summarizer_node(state: GraphState):
    """Refine and format the final answer"""
    answer = state.get("answer")
    
    if not answer or answer == "No relevant documents found.":
        return {"answer": answer}
    
    prompt = ChatPromptTemplate.from_template(
        """Refine and format the following answer to be clear, concise, and professional.
        Keep the key information but improve readability.

        Answer to refine:
        {answer}

        Refined answer:"""
    )
    
    chain = prompt | llm | parser
    refined = chain.invoke({"answer": answer})
    
    print("Answer summarized and refined")
    
    return {"answer": refined}


In [60]:
def memory_node(state: GraphState):
    """Manage conversation memory - save current interaction"""
    query = state["query"]
    answer = state.get("answer", "")
    
    # Save the current interaction to memory
    memory.save_context(
        {"query": query},
        {"answer": answer}
    )
    
    # Get updated chat history
    chat_history = memory.chat_memory.messages
    
    print(f"Memory updated. Total messages in history: {len(chat_history)}")
    
    return {
        "chat_history": chat_history
    }


## Routing Logic


In [61]:
def route_after_retrieval(state: GraphState) -> Literal["rag", "web_search"]:
    """Route to RAG or web search based on document retrieval"""
    if state.get("use_rag"):
        print("Routing to RAG")
        return "rag"
    else:
        print("Routing to Web Search")
        return "web_search"


## Graph Construction


In [62]:
# Build the graph with memory
graph = StateGraph(GraphState)

# Add nodes
graph.add_node("retriever", retriever_node)
graph.add_node("rag", rag_node)
graph.add_node("web_search", web_search_node)
graph.add_node("summarizer", summarizer_node)
graph.add_node("memory", memory_node)

# Set entry point
graph.set_entry_point("retriever")

# Add conditional routing after retrieval
graph.add_conditional_edges(
    "retriever",
    route_after_retrieval,
    {
        "rag": "rag",
        "web_search": "web_search"
    }
)

# Both RAG and web search go to summarizer
graph.add_edge("rag", "summarizer")
graph.add_edge("web_search", "summarizer")

# Summarizer goes to memory management
graph.add_edge("summarizer", "memory")

# Memory goes to END
graph.add_edge("memory", END)

# Compile the graph
app = graph.compile()

print("AgenticRAG with ConversationBufferWindowMemory is ready!")


AgenticRAG with ConversationBufferWindowMemory is ready!


## Memory Management Utilities


In [63]:
def clear_memory():
    """Clear the conversation memory"""
    memory.clear()
    print("Memory cleared!")

def show_memory():
    """Display current memory contents"""
    messages = memory.chat_memory.messages
    print(f"Current memory contains {len(messages)} messages:")
    for i, msg in enumerate(messages):
        if isinstance(msg, HumanMessage):
            print(f"  {i+1}. Human: {msg.content[:100]}...")
        elif isinstance(msg, AIMessage):
            print(f"  {i+1}. Assistant: {msg.content[:100]}...")

def get_memory_summary():
    """Get a summary of memory usage"""
    messages = memory.chat_memory.messages
    return {
        "total_messages": len(messages),
        "human_messages": len([m for m in messages if isinstance(m, HumanMessage)]),
        "ai_messages": len([m for m in messages if isinstance(m, AIMessage)]),
        "memory_window_size": memory.k
    }

def chat_with_memory(query: str):
    """Simple interface to chat with the RAG system"""
    print(f"\nYou: {query}")
    result = app.invoke({"query": query})
    print(f"Assistant: {result['answer']}")
    print(f"Source: {result['source']}")
    return result

print("Memory management utilities loaded.")


Memory management utilities loaded.


## Testing the System


In [64]:
# Test the system
print("Testing AgenticRAG with ConversationBufferWindowMemory")
print("=" * 60)

# Test 1: First question
print("\nTest 1: First question")
query1 = "What is Management Information System?"
result1 = app.invoke({"query": query1})
print(f"Answer: {result1['answer'][:200]}...")
print(f"Source: {result1['source']}")

# Test 2: Follow-up question (should use memory)
print("\nTest 2: Follow-up question (using memory)")
query2 = "What are its main components?"
result2 = app.invoke({"query": query2})
print(f"Answer: {result2['answer'][:200]}...")
print(f"Source: {result2['source']}")

# Test 3: Another follow-up
print("\nTest 3: Another follow-up question")
query3 = "How does it relate to CRM?"
result3 = app.invoke({"query": query3})
print(f"Answer: {result3['answer'][:200]}...")
print(f"Source: {result3['source']}")

print("\nMemory integration test completed!")


Testing AgenticRAG with ConversationBufferWindowMemory

Test 1: First question
Retriever found 5 relevant docs from 5 total
Routing to RAG
RAG answered using 5 documents with memory context
Answer summarized and refined
Memory updated. Total messages in history: 2
Answer: **Management Information System (MIS)**

A Management Information System (MIS) is a comprehensive information system designed to support decision-making and facilitate the coordination, control, analy...
Source: RAG (Documents: test.pdf)

Test 2: Follow-up question (using memory)
Retriever found 0 relevant docs from 5 total
Routing to Web Search
Executing web search with memory context...
Answer summarized and refined
Memory updated. Total messages in history: 4
Answer: **Computer Components: Hardware and Software**

**Hardware Components:**

1. **Central Processing Unit (CPU):** The brain of the computer, responsible for executing instructions.
2. **Motherboard:** T...
Source: Web Search (SerpAPI)

Test 3: Another fo

In [65]:
# Display memory status
print("Memory Status:")
memory_status = get_memory_summary()
for key, value in memory_status.items():
    print(f"  {key}: {value}")

print("\nExample conversation with memory:")
print("=" * 40)

# You can now use chat_with_memory() for interactive conversations
# The system will remember the last 3 exchanges (6 messages total)


Memory Status:
  total_messages: 6
  human_messages: 3
  ai_messages: 3
  memory_window_size: 6

Example conversation with memory:


## Usage Examples


In [66]:
# Example usage:
# chat_with_memory("What is Management Information System?")
# chat_with_memory("What are its main components?")
# chat_with_memory("How does it relate to CRM?")

# Check memory status
# show_memory()
# get_memory_summary()

# Clear memory when needed
# clear_memory()

print("System ready for use!")


System ready for use!
